In [ ]:
#import 
import logging
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import KNNImputer
from sklearn.neighbors import KNeighborsClassifier
from umap import UMAP
import matplotlib.pyplot as plt
from reval.best_nclust_cv import FindBestClustCV

In [ ]:
main_path = '/Users/vmandelli/OneDrive - Fondazione Istituto Italiano Tecnologia/vine_old_edition'
data_path = os.path.join(main_path,"data","tidy","vineland_ed_corrected")
results_path = os.path.join(main_path,"results")
plot_path = os.path.join(main_path,"plot")

In [ ]:
# load train set
file = 'NDA_tr_format_corrected_old.csv'
X_tr_ = pd.read_csv(os.path.join(data_path,file),
                        header=0,
                        #skiprows=[1],
                        low_memory=False,
                        index_col='subjectkey')
# load test set
file = 'NDA_ts_format_corrected_old.csv'
X_ts_ = pd.read_csv(os.path.join(data_path,file),
                        header=0,
                        #skiprows=[1],
                        low_memory=False,
                        index_col='subjectkey')
#keep track of the original
TR_original = X_tr_.copy()
TS_original = X_ts_.copy()

In [ ]:
vabs_names = ['communicationdomain_totalb','livingskillsdomain_totalb',
              'socializationdomain_totalb']
col2use=vabs_names

In [ ]:
# preprocessing

# imputation
impute = KNNImputer(n_neighbors=5)

# scaling  
Scaler = StandardScaler()

#umap
Umap = UMAP(n_neighbors=30, min_dist=0.0, n_components=2, random_state=42)

# impute then scale than UMAP
X_tr_prepr = pd.DataFrame(Umap.fit_transform(Scaler.fit_transform(impute.fit_transform(X_tr_[col2use]))), index = X_tr_.index)
X_ts_prepr = pd.DataFrame(Umap.transform(Scaler.transform(impute.transform(X_ts_[col2use]))), index = X_ts_.index)



In [ ]:
print(len(X_tr_.index))
print(len(X_ts_.index))

In [ ]:
plt.figure(figsize=(15,6))
plt.scatter(X_tr_prepr[[0]], X_tr_prepr[[1]],alpha=0.2)

In [ ]:
def Gridsearch(X_tr,strat_vect):

    
    # GRID SEARCH
    
    # initialize impty saving matrices
    #metric_df = pd.DataFrame()
    metric_dict ={'fold': [],"n_neigh":[],'ncl':[],'stab':[],'err':[]}
    
    
    # define params to test
    # KMEANS
    clust = KMeans(random_state=42)
    vect_hyper = [5,15,30]
    
    
    vect_fold = [2]
    
    # run the grid search
    for f in vect_fold:
        for n in vect_hyper:
            logging.info(f"fold:{f} -- nighbors:{n}")
            clf = KNeighborsClassifier(n_neighbors=n)
            #clf = RandomForestClassifier(n_estimators=n)
            relval = FindBestClustCV(s=clf, c=clust, nfold=f, nclust_range=list(range(2,11,1)), nrand=100)
            metric, ncl,_ = relval.best_nclust(X_tr,iter_cv=100, strat_vect = strat_vect) 
            metric_dict['fold'].append(f)
            metric_dict['n_neigh'].append(n)
            metric_dict['ncl'].append(ncl)
            metric_dict['stab'].append(metric['val'][ncl][0])
            metric_dict['err'].append(metric['val'][ncl][1][1])

    metric_df = pd.DataFrame(metric_dict)

    return metric_df


In [ ]:
len(X_tr_prepr.index.unique())

In [ ]:
X_tr_

In [ ]:
#define stract vect
strat_vect = X_tr_['sex'] + X_tr_['quantile_age'].astype("str")+X_tr_['format']
strat_vect

In [ ]:
# define dataset
X_tr = X_tr_prepr
X_ts = X_ts_prepr
X_tr_indx = X_tr.index
strat_vect = strat_vect

In [ ]:
print(len(X_tr_prepr))
print(len(X_tr_indx))
print(len(strat_vect))

In [ ]:
# run grid search
metric_df_VABS_old = Gridsearch(X_tr , strat_vect)

In [ ]:
mean_stab_err = list(metric_df_VABS_old[["stab","err"]].apply(np.mean,1))
idx = mean_stab_err.index(min(mean_stab_err))
num_neigh = pd.to_numeric(metric_df_VABS_old.loc[idx]['n_neigh']).astype(np.int64)

In [ ]:
#import pandas as pd
metric_df_VABS_old.to_csv(os.path.join(results_path,'Grid_search_051022.csv'))

In [ ]:
#metric_df_VABS_old

In [ ]:
num_neigh

In [ ]:
# run reval (only the train)
# classifcation algorhythm
clf = KNeighborsClassifier(n_neighbors= num_neigh)      #KNN  
# cluster algorhythm
clust = KMeans(random_state=42)                #Kmeans

# define dataset
X_tr = X_tr_prepr
X_ts = X_ts_prepr
X_tr_indx = X_tr.index
strat_vect = strat_vect[X_tr_indx]


# initialize the class
relval = FindBestClustCV(s=clf, c=clust, nfold=2, nclust_range=list(range(2,11,1)), nrand=100)
    
# train
metric, ncl,_ = relval.best_nclust(X_tr,iter_cv=100, strat_vect = strat_vect)
print(metric)
print(ncl)

out = relval.evaluate(X_tr, X_ts, ncl) # riscrivi NCL qui! il best!

logging.info(f"Training ACC: {out.train_acc}, Test ACC: {out.test_acc}")

In [ ]:
ncl

In [ ]:
from reval.visualization import plot_metrics
plot_metrics(metric)

In [ ]:
#umap plot train
import matplotlib.pyplot as plt
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']

plt.scatter(X_tr_prepr[[0]], X_tr_prepr[[1]], color=[colors[lab] for lab in out.train_cllab])

In [ ]:
plt.scatter(X_ts_prepr[[0]], X_ts_prepr[[1]], color=[colors[lab] for lab in out.test_cllab])

In [ ]:
X_tr_['cluster_domain']=  out.train_cllab
X_tr_["TR_TS"] = "TR"

X_ts_['cluster_domain']=  out.test_cllab
X_ts_["TR_TS"] = "TS"

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
dataset_trts_cl = X_tr_.append(X_ts_)
cols = ['interview_age', 'communicationdomain_totalb','livingskillsdomain_totalb','socializationdomain_totalb','composite_totalb']
for c in cols:
    _data_train = dataset_trts_cl.loc[dataset_trts_cl['TR_TS']=='TR',]
    plt.figure()
    sns.boxplot(x="cluster_domain", y=c, data=_data_train)


In [ ]:
X_tr_split.groupby(['cluster_domain']).count()

In [ ]:
X_ts_split.groupby(['cluster_domain']).mean()

In [ ]:
for table in metric.keys():
    data = pd.DataFrame(columns=['k','value','CI'])
    for row in metric[table]:
        _dict_append = {'k':row,
                     'value':metric[table][row][1][0],
                     'CI':metric[table][row][1][1]}
        data= data.append(_dict_append,ignore_index=True)
        
    data.to_csv(os.path.join(results_path,'_'.join([table,'metric_for_plot_051022.csv'])))

In [ ]:
#save umap values for future works!
X_tr_2save = X_tr.copy()
X_ts_2save = X_ts.copy()
X_tr_2save['tr_ts']='tr'
X_tr_2save['cluster_domain']=out.train_cllab
X_ts_2save['tr_ts']='ts'
X_ts_2save['cluster_domain']=out.test_cllab
UMAP_values_2save = X_tr_2save.append(X_ts_2save)
UMAP_values_2save.to_csv(os.path.join(results_path,"UMAP_values_VABSold_051022.csv"))

In [ ]:
X_tr_2save

In [ ]:
#save results
TR_original['cluster_domain']=  out.train_cllab
TR_original["TR_TS"] = "TR"

TS_original['cluster_domain']=  out.test_cllab
TS_original["TR_TS"] = "TS"
dataset_trts_cl = TR_original.append(TS_original)
dataset_trts_cl.to_csv(os.path.join(results_path,"VABS_old_withcl_051022_withgridsearch.csv"))

In [ ]:
dataset_trts_cl